# Bengaluru Google Routes Data Collector — Utility Notebook

Collect route-duration and traffic-aware estimates from Google Maps Platform whenever you run this notebook.

## What this utility does
- Calls Google Routes API for a configurable list of Bengaluru/suburb routes.
- Lets you choose a collection date, weekday, season, event window and departure time.
- Saves each response as a collection record and maintains a deduplicated master CSV.
- If the same route + requested departure slot + weekday + season + date range is collected again, the existing record is **overwritten** rather than duplicated.
- Stores the API collection timestamp separately from the requested traffic/departure date.

## Important limitations
- For driving routes, Google Routes API does **not** provide arbitrary past-date traffic retrieval. Use a current/future departure time only.
- `duration` with `TRAFFIC_AWARE` / `TRAFFIC_AWARE_OPTIMAL` is an estimate for the requested departure time, not a measured vehicle count.
- `staticDuration` is a traffic-unaware route duration, not a historical traffic observation.
- This utility cannot provide actual number of vehicles or commuters on a road.
- Review Google Maps Platform billing, API terms, and applicable caching/storage restrictions before running and persisting results.

## Setup
1. Enable Routes API and billing in your Google Cloud project.
2. Set `GOOGLE_MAPS_API_KEY` in your environment; never paste a real key into a shared notebook.
3. Run cells from top to bottom.
4. Set the requested departure date/time and sampling labels.
5. Set `RUN_COLLECTION = True` only when ready to make billable requests.

Install dependencies if required:
`%pip install pandas requests`

In [ ]:
import os
import time
import json
import hashlib
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import requests

TZ = ZoneInfo("Asia/Kolkata")
GOOGLE_ROUTES_ENDPOINT = "https://routes.googleapis.com/directions/v2:computeRoutes"

# API key is read from the environment.
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

# Storage location; adjust if you want another project folder.
DATA_DIR = Path("bengaluru_google_traffic_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = DATA_DIR / "google_routes_master.csv"
RUN_LOG_CSV = DATA_DIR / "collection_run_log.csv"

print("API key configured:", bool(GOOGLE_MAPS_API_KEY))
print("Data directory:", DATA_DIR.resolve())

## 1. Configure the collection request

For a new sample, change the date/time and labels below. For a historical seasonal baseline, Google cannot retrieve arbitrary past-day traffic; set a current/future departure date and use `season_label` only as an analysis grouping label, not as proof of historical conditions.

In [ ]:
# Requested departure date/time in Bengaluru local time.
# For traffic-aware driving routes, use current or future times.
DEPARTURE_DATETIME_LOCAL = (datetime.now(TZ) + timedelta(hours=2)).replace(microsecond=0)

# Labels used for comparison. They are metadata supplied by you, not Google measurements.
SEASON_LABEL = "monsoon"       # e.g. summer / monsoon / winter / other
EVENT_LABEL = "regular_week"   # e.g. regular_week / pre_deepavali / post_deepavali
DATE_RANGE_LABEL = "adhoc_7day_sample"  # e.g. summer_2026_week1 / pre_deepavali_2026

# Requested time slot label for deduplication; e.g. 08:00, 13:00, 17:00, 20:00.
DEPARTURE_SLOT = DEPARTURE_DATETIME_LOCAL.strftime("%H:%M")

# Route preference: TRAFFIC_AWARE or TRAFFIC_AWARE_OPTIMAL.
# Use a valid traffic-aware routing preference for departure-time estimates.
ROUTING_PREFERENCE = "TRAFFIC_AWARE"

# Prevent accidental billable API calls until explicitly enabled.
RUN_COLLECTION = False

print("Departure:", DEPARTURE_DATETIME_LOCAL.isoformat())
print("Weekday:", DEPARTURE_DATETIME_LOCAL.strftime("%A"))
print("Season:", SEASON_LABEL, "| Event:", EVENT_LABEL, "| Date range:", DATE_RANGE_LABEL)
print("Departure slot:", DEPARTURE_SLOT)

## 2. Define routes

Edit, add or remove routes as needed. Route endpoints are address strings; Google may resolve them to slightly different locations, so use stable, precise landmarks where possible.

In [ ]:
routes = pd.DataFrame([
    {"route_id":"ORR_MARATHAHALLI_BELLANDUR", "origin":"Marathahalli Bridge, Bengaluru, Karnataka", "destination":"Bellandur, Bengaluru, Karnataka", "corridor":"ORR / IT corridor"},
    {"route_id":"WHITEFIELD_MARATHAHALLI", "origin":"ITPL, Whitefield, Bengaluru, Karnataka", "destination":"Marathahalli Bridge, Bengaluru, Karnataka", "corridor":"East Bengaluru"},
    {"route_id":"ELECTRONIC_CITY_SILK_BOARD", "origin":"Electronic City Phase 1, Bengaluru, Karnataka", "destination":"Silk Board Junction, Bengaluru, Karnataka", "corridor":"South Bengaluru"},
    {"route_id":"SARJAPUR_BELLANDUR", "origin":"Sarjapur, Bengaluru, Karnataka", "destination":"Bellandur, Bengaluru, Karnataka", "corridor":"South-east suburb"},
    {"route_id":"HEBBAL_MG_ROAD", "origin":"Hebbal Flyover, Bengaluru, Karnataka", "destination":"MG Road Metro Station, Bengaluru, Karnataka", "corridor":"North to central"},
    {"route_id":"YELAHANKA_HEBBAL", "origin":"Yelahanka, Bengaluru, Karnataka", "destination":"Hebbal Flyover, Bengaluru, Karnataka", "corridor":"North Bengaluru"},
    {"route_id":"KENGERI_MYSORE_ROAD", "origin":"Kengeri, Bengaluru, Karnataka", "destination":"Mysore Road Metro Station, Bengaluru, Karnataka", "corridor":"West Bengaluru"},
    {"route_id":"HOSKOTE_WHITEFIELD", "origin":"Hoskote, Karnataka", "destination":"Whitefield, Bengaluru, Karnataka", "corridor":"Eastern suburb"},
])
display(routes)

## 3. Google Routes API request

The field mask requests only route distance and duration fields. The API response gives a route estimate, not traffic volume. This collector uses the requested departure time, which must be current/future for driving traffic-aware routes.

In [ ]:
def duration_seconds(value):
    if value is None:
        return None
    value = str(value).strip()
    return float(value[:-1]) if value.endswith("s") else None


def request_route(origin, destination, departure_time, routing_preference="TRAFFIC_AWARE"):
    if not GOOGLE_MAPS_API_KEY:
        raise RuntimeError(
            "GOOGLE_MAPS_API_KEY is not set. Configure it in your environment before running."
        )

    departure_time = pd.Timestamp(departure_time)
    if departure_time.tzinfo is None:
        departure_time = departure_time.tz_localize(TZ)
    departure_time = departure_time.tz_convert("UTC")

    body = {
        "origin": {"address": origin},
        "destination": {"address": destination},
        "travelMode": "DRIVE",
        "routingPreference": routing_preference,
        "departureTime": departure_time.isoformat().replace("+00:00", "Z")
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": "routes.distanceMeters,routes.duration,routes.staticDuration"
    }
    response = requests.post(
        GOOGLE_ROUTES_ENDPOINT, headers=headers, json=body, timeout=40
    )
    response.raise_for_status()
    payload = response.json()
    route_list = payload.get("routes", [])
    if not route_list:
        return {
            "status":"no_route", "distance_meters":None,
            "traffic_duration_seconds":None, "static_duration_seconds":None,
            "raw_status":"empty_routes"
        }

    route = route_list[0]
    return {
        "status":"ok",
        "distance_meters":route.get("distanceMeters"),
        "traffic_duration_seconds":duration_seconds(route.get("duration")),
        "static_duration_seconds":duration_seconds(route.get("staticDuration")),
        "raw_status":"ok"
    }

## 4. Collect and overwrite duplicate samples

Deduplication key:
`route_id + departure_date + departure_slot + weekday + season_label + event_label + date_range_label`.

If a matching key exists in the master CSV, it is replaced with the newest response. Otherwise, a new row is appended. The API collection timestamp is retained so you can audit when the estimate was retrieved.

Note: overwriting the same key is appropriate for refreshing an estimate, but repeated API calls may return different values as traffic conditions change. If you want to preserve every run for audit purposes, use the run log as a separate append-only record.

In [ ]:
def load_master():
    if MASTER_CSV.exists():
        return pd.read_csv(MASTER_CSV)
    return pd.DataFrame()


def collect_routes():
    departure = pd.Timestamp(DEPARTURE_DATETIME_LOCAL)
    if departure.tzinfo is None:
        departure = departure.tz_localize(TZ)
    else:
        departure = departure.tz_convert(TZ)

    now = datetime.now(TZ)
    if departure < now - timedelta(minutes=1):
        raise ValueError(
            "Google driving traffic-aware requests cannot retrieve arbitrary past departure times. "
            "Choose a current or future departure time."
        )

    sample_date = departure.strftime("%Y-%m-%d")
    weekday = departure.strftime("%A")
    slot = departure.strftime("%H:%M")
    collected_at = now.isoformat()

    records = []
    for _, route in routes.iterrows():
        try:
            result = request_route(
                route["origin"], route["destination"], departure,
                routing_preference=ROUTING_PREFERENCE
            )
            record = {
                "route_id":route["route_id"],
                "origin":route["origin"],
                "destination":route["destination"],
                "corridor":route["corridor"],
                "departure_datetime_local":departure.isoformat(),
                "departure_date":sample_date,
                "departure_slot":slot,
                "weekday":weekday,
                "season_label":SEASON_LABEL,
                "event_label":EVENT_LABEL,
                "date_range_label":DATE_RANGE_LABEL,
                "routing_preference":ROUTING_PREFERENCE,
                "distance_meters":result.get("distance_meters"),
                "traffic_duration_seconds":result.get("traffic_duration_seconds"),
                "static_duration_seconds":result.get("static_duration_seconds"),
                "status":result.get("status"),
                "evidence_type":"google_api_traffic_aware_route_estimate",
                "is_actual_vehicle_count":False,
                "is_actual_historical_observation":False,
                "collected_at_local":collected_at
            }
            records.append(record)
            print(route["route_id"], "->", result.get("status"))
        except Exception as exc:
            records.append({
                "route_id":route["route_id"],
                "origin":route["origin"],
                "destination":route["destination"],
                "corridor":route["corridor"],
                "departure_datetime_local":departure.isoformat(),
                "departure_date":sample_date,
                "departure_slot":slot,
                "weekday":weekday,
                "season_label":SEASON_LABEL,
                "event_label":EVENT_LABEL,
                "date_range_label":DATE_RANGE_LABEL,
                "routing_preference":ROUTING_PREFERENCE,
                "status":"error",
                "error_message":f"{type(exc).__name__}: {exc}",
                "evidence_type":"google_api_traffic_aware_route_estimate",
                "is_actual_vehicle_count":False,
                "is_actual_historical_observation":False,
                "collected_at_local":collected_at
            })
            print(route["route_id"], "-> ERROR:", exc)
        time.sleep(0.15)

    fresh = pd.DataFrame(records)
    master = load_master()

    dedupe_keys = [
        "route_id", "departure_date", "departure_slot", "weekday",
        "season_label", "event_label", "date_range_label"
    ]

    if master.empty:
        combined = fresh
    else:
        # Add missing columns if schemas changed between runs.
        for col in fresh.columns:
            if col not in master.columns:
                master[col] = pd.NA
        for col in master.columns:
            if col not in fresh.columns:
                fresh[col] = pd.NA
        fresh = fresh[master.columns]
        # Remove existing records with keys that are being refreshed.
        new_keys = fresh[dedupe_keys].astype(str).drop_duplicates()
        master_keys = master[dedupe_keys].astype(str)
        refreshed = master_keys.merge(
            new_keys, on=dedupe_keys, how="left", indicator=True
        )["_merge"].eq("both")
        master = master.loc[~refreshed].copy()
        combined = pd.concat([master, fresh], ignore_index=True)

    combined.to_csv(MASTER_CSV, index=False)

    # Append-only run log records each collection attempt.
    log_rows = fresh.copy()
    log_rows["run_logged_at_local"] = datetime.now(TZ).isoformat()
    if RUN_LOG_CSV.exists():
        old_log = pd.read_csv(RUN_LOG_CSV)
        log_rows = pd.concat([old_log, log_rows], ignore_index=True)
    log_rows.to_csv(RUN_LOG_CSV, index=False)

    print(f"Saved master: {MASTER_CSV.resolve()}")
    print(f"Saved run log: {RUN_LOG_CSV.resolve()}")
    print("Records in master:", len(combined))
    return fresh, combined


if RUN_COLLECTION:
    latest_run, master_data = collect_routes()
    display(latest_run)
else:
    print("Collection disabled. Review the request, then set RUN_COLLECTION = True.")

## 5. Review collected data

Use these views after a successful collection. Duration is measured in seconds; convert to minutes for interpretation. Never interpret duration as a count of vehicles or commuters.

In [ ]:
if MASTER_CSV.exists():
    master_data = pd.read_csv(MASTER_CSV)
    display(master_data.tail(20))

    if "traffic_duration_seconds" in master_data:
        master_data["traffic_duration_minutes"] = (
            pd.to_numeric(master_data["traffic_duration_seconds"], errors="coerce") / 60
        )
        display(
            master_data.sort_values("collected_at_local", ascending=False)
            [["route_id","departure_date","departure_slot","weekday",
              "season_label","event_label","traffic_duration_minutes",
              "static_duration_seconds","status"]]
            .head(30)
        )
else:
    print("No data collected yet.")

## 6. Operational reminders

- Run this notebook at each desired sampling time. It does not run in the background.
- For a seven-day comparison, run the same routes at the same time slots on each of the seven days.
- The overwrite key includes the actual departure date as well as weekday, season and date-range labels. This prevents a Tuesday from overwriting a different Tuesday in another week. Re-running the same date/slot/labels refreshes that record.
- If you intentionally want to overwrite records for the same weekday and season across different dates, remove `departure_date` from `dedupe_keys` in the collection cell. This will collapse different dates together, so it is not recommended for day-by-day research.
- Google API response values and storage are subject to current Google Maps Platform terms and billing. Check current terms before long-term retention.